# 01. Parameter-Efficient Fine-Tuning: LoRA, rsLoRA, LoRA+, and DoRA

- **Category:** 15_compression_peft_and_speculative_inference / 03_parameter_efficient_finetuning_peft
- **Lineage & Citations:** Hu et al. (ICLR 2022, *LoRA: Low-Rank Adaptation of Large Language Models*); Kalajdzievski (2023, *Rank-Stabilized LoRA*); Hayou et al. (ICML 2024, *LoRA+: Efficient Low Rank Adaptation of Large Models*); Liu et al. (ICML 2024, *DoRA: Weight-Decomposed Low-Rank Adaptation*).
- **Purpose:** Implement standard LoRA, rank-stabilized rsLoRA, asymmetric learning-rate dynamics (LoRA+), and magnitude-direction decoupled adaptation (DoRA).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Model Merging Slerp Task Arithmetic TIES DARE and Evolutionary](../02_adaptive_kv_speculative_and_merging/05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.ipynb) $\to$ `🔗 Downstream Syntheses:` [PEFT SVD and Adaptive PiSSA Adalora Vera](02_peft_svd_and_adaptive_pissa_adalora_vera.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Never train high-rank adapters ($r \ge 32$) with standard $1/r$ LoRA scaling; gradient update collapse will stall optimization (use rsLoRA instead). Do not apply DoRA if zero-overhead inference folding is strictly required, unless normalized magnitude matrices are pre-computed.

## Mathematical Foundations & Governing Equations

### 1. Standard LoRA & Weight Folding
$$h = x W_0 + \frac{\alpha}{r} (x A) B, \quad W_{\text{merged}} = W_0 + \frac{\alpha}{r} A B$$

### 2. Rank-Stabilized LoRA (rsLoRA)
$$\gamma_{\text{rsLoRA}} = \frac{\alpha}{\sqrt{r}}, \quad \lim_{r \to \infty} \|\nabla_{\Delta W} \mathcal{L}\|_F = \Theta(1)$$

### 3. LoRA+ Asymmetric Optimization
$$\eta_B = \lambda \eta_A, \quad \lambda \approx 16$$

### 4. Weight-Decomposed Low-Rank Adaptation (DoRA)
$$V = W_0 + \frac{\alpha}{r} A B, \quad W_{\text{dora}} = m \odot \frac{V}{\|V\|_c}, \quad h = x W_{\text{dora}}$$

### Complexity
Reduces trainable parameters from $\mathcal{O}(d^2)$ to $\mathcal{O}(2 d r)$ ($>99.8\%$ reduction), while weight folding eliminates inference latency.

In [ ]:
import math
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/01_peft_lora_rslora_loraplus_and_dora.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x, w, b):
    return x @ w + b

def lora_forward(x, w0, a, b, alpha=4.0, r=2):
    return x @ w0 + (alpha / float(r)) * ((x @ a) @ b)

def rslora_forward(x, w0, a, b, alpha=4.0, r=2):
    return x @ w0 + (alpha / math.sqrt(float(r))) * ((x @ a) @ b)

def dora_forward(x, w0, a, b, m, alpha=4.0, r=2, eps=1e-8):
    scale = alpha / float(r)
    v = w0 + scale * (a @ b)
    col_norm = torch.norm(v, p=2, dim=0, keepdim=True).clamp_min(eps)
    w_dora = (m / col_norm) * v
    return x @ w_dora

In [ ]:
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-6

l_out = lora_forward(weights["x"], weights["w0"], weights["a"], weights["b"], 4.0, 2)
assert float(torch.max(torch.abs(l_out - weights["lora_out"]))) < 1e-5

rs_out = rslora_forward(weights["x"], weights["w0"], weights["a"], weights["b"], 4.0, 2)
assert float(torch.max(torch.abs(rs_out - weights["rslora_out"]))) < 1e-5

d_out = dora_forward(weights["x"], weights["w0"], weights["a"], weights["b"], weights["m"], 4.0, 2)
assert float(torch.max(torch.abs(d_out - weights["dora_out"]))) < 1e-5
print("All 4 PEFT adapter architectures match golden checkpoints (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_lora = benchmark_fn(lambda: lora_forward(weights["x"], weights["w0"], weights["a"], weights["b"]), iters=100, name="LoRA Forward (r=2)")
b_dora = benchmark_fn(lambda: dora_forward(weights["x"], weights["w0"], weights["a"], weights["b"], weights["m"]), iters=100, name="DoRA Forward (Norm Decoupled)")
print(format_benchmark_table([b_lora, b_dora]))

d_in, d_out, r = 4096, 4096, 8
base_params = d_in * d_out
lora_params = (d_in + d_out) * r
print(f"7B Linear Layer Parameters: Full FT={base_params:,} vs LoRA={lora_params:,} ({(lora_params/base_params)*100:.2f}% parameters)")

In [ ]:
# Pathological Misuse: Standard LoRA scaling 1/r causes gradient vanishing as rank scales!
alpha = 16.0
ranks = [4, 16, 64]
lora_scales = [alpha / r for r in ranks]
rslora_scales = [alpha / math.sqrt(r) for r in ranks]

# Ratio of scale drop from r=4 to r=64
lora_drop = lora_scales[0] / lora_scales[-1]      # Drops by 16x!
rslora_drop = rslora_scales[0] / rslora_scales[-1]  # Drops by only 4x!

assert lora_drop == 16.0, f"Standard LoRA drops update magnitude by {lora_drop}x!"
assert rslora_drop == 4.0, f"rsLoRA preserves gradient scale (only {rslora_drop}x drop)!"
print(f"Verified: At r=64, LoRA collapses scale by {lora_drop:.0f}x while rsLoRA maintains {rslora_drop:.0f}x stability.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Zero-Latency Invariance**: Pre-folding low-rank adapters ($W_0 + \frac{\alpha}{r} AB$) achieves identical inference latency to the base pretrained model.
2. **The Rank Scaling Law**: Use rsLoRA ($1/\sqrt{r}$) whenever fine-tuning with rank $r \ge 32$ to prevent gradient norm vanishing.
3. **DoRA Decoupling**: Decoupling magnitude from direction updates enables parameter-efficient tuning to match or exceed full fine-tuning accuracy.

---
### Cross-Domain Generalisation Matrix
- **Control Systems:** Low-rank feedback controllers adapt high-dimensional dynamical systems with minimal parameter gains.
- **Diffusion Models:** LoRA fine-tunes text-to-image UNet and DiT cross-attention layers for subject-driven generation in 50MB checkpoints.
- **Quantum Chemistry:** Low-rank perturbation theory (MP2) adjusts Hartree-Fock wavefunctions with minimal correction matrices.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_peft_lora_rslora_loraplus_and_dora.md](../../../markdown/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/01_peft_lora_rslora_loraplus_and_dora.md)
- **PyTorch 2.x Module:** [01_peft_lora_rslora_loraplus_and_dora.py](../../../pytorch2/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/01_peft_lora_rslora_loraplus_and_dora.py)
- **Functional JAX Module:** [01_peft_lora_rslora_loraplus_and_dora.py](../../../jax/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/01_peft_lora_rslora_loraplus_and_dora.py)
- **Native Rust Candle Track:** [01_peft_lora_rslora_loraplus_and_dora.rs](../../../candle/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/01_peft_lora_rslora_loraplus_and_dora.rs)
- **Hardware Kernel (CPU / WGSL):** [01_peft_lora_rslora_loraplus_and_dora.rs](../../../kernels/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/01_peft_lora_rslora_loraplus_and_dora.rs)
- **Golden SafeTensors:** [01_peft_lora_rslora_loraplus_and_dora.safetensors](../../../weights/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/01_peft_lora_rslora_loraplus_and_dora.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 05. Model Merging Slerp Task Arithmetic TIES DARE and Evolutionary](../02_adaptive_kv_speculative_and_merging/05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 02. PEFT SVD and Adaptive PiSSA Adalora Vera](02_peft_svd_and_adaptive_pissa_adalora_vera.ipynb) &rarr;